# Assignment 10: Text Analytics
## Text Preprocessing, Analysis & Sentiment using NLTK, TextBlob & WordCloud

**Name:** ___________ | **Class:** T.E. | **Roll No:** ___________

## Problem Statement

To perform text preprocessing and analytics on textual data: tokenization (word and sentence), stop word removal, stemming and lemmatization, lowercasing and punctuation removal, word frequency analysis, WordCloud generation, N-gram analysis, and sentiment analysis using TextBlob and VADER.

## Theory

**Text Analytics Pipeline:**
1. **Lowercasing** — normalize case ("Hello" = "hello")
2. **Punctuation Removal** — remove noise characters
3. **Tokenization** — split text into words/sentences
4. **Stop Word Removal** — filter out common words ("the", "is", "and")
5. **Stemming/Lemmatization** — reduce words to base form

**Stemming vs Lemmatization:**
- Stemming: Fast, rule-based chopping ("running" → "run", "studies" → "studi")
- Lemmatization: Dictionary-based, returns real words ("better" → "good")

**Sentiment Analysis:**
- **VADER:** Rule-based, specifically tuned for social media
- **TextBlob:** Returns polarity (-1 to +1) and subjectivity (0 to 1)

In [1]:
# Step 1: Import Required Libraries
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import FreqDist, bigrams, trigrams
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import pandas as pd
import re
import string

# Download NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

print('All NLTK resources downloaded successfully!')

ModuleNotFoundError: No module named 'textblob'

In [ ]:
# Step 2: Create Sample Text Corpus
sample_text = '''
Data science is an exciting field! Data scientists use Python and machine learning
to analyze data and make predictions. The field of data science continues to grow rapidly.
Machine learning algorithms are becoming more sophisticated every year. Python is the most
popular programming language for data science and artificial intelligence.
Data scientists love working with large datasets and building predictive models.
The future of data science looks bright with advances in deep learning and neural networks.
Statistics is the foundation of data science. Understanding probability and distributions
is essential for any aspiring data scientist. Big data technologies like Spark and Hadoop
enable processing of massive datasets that traditional tools cannot handle.
'''
print('Sample text loaded:')
print(sample_text[:300] + '...')

In [ ]:
# Step 3: Tokenization — Sentence and Word Level
sentences = nltk.sent_tokenize(sample_text)
words = nltk.word_tokenize(sample_text)

print(f'Number of sentences: {len(sentences)}')
print(f'Number of word tokens: {len(words)}')
print()
print('Sentences:')
for i, sent in enumerate(sentences, 1):
    print(f'  {i}. {sent.strip()}')
print()
print('First 20 word tokens:')
print(f'  {words[:20]}')

In [ ]:
# Step 4: Lowercasing, Punctuation Removal, and Stop Word Removal
# Lowercase
words_lower = [w.lower() for w in words]

# Remove punctuation and non-alphabetic tokens
words_alpha = [w for w in words_lower if w.isalpha()]

# Remove stop words
stop_words = set(stopwords.words('english'))
words_filtered = [w for w in words_alpha if w not in stop_words]

print(f'Original tokens: {len(words)}')
print(f'After lowercasing + alpha only: {len(words_alpha)}')
print(f'After stop word removal: {len(words_filtered)}')
print(f'\nStop words removed (sample): {list(stop_words)[:15]}')
print(f'\nFiltered word tokens (first 30):')
print(f'  {words_filtered[:30]}')

In [ ]:
# Step 5: Stemming and Lemmatization
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Apply stemming
stemmed = [stemmer.stem(w) for w in words_filtered]

# Apply lemmatization
lemmatized = [lemmatizer.lemmatize(w) for w in words_filtered]

# Show comparison
comparison = pd.DataFrame({
    'Original': words_filtered[:15],
    'Stemmed': stemmed[:15],
    'Lemmatized': lemmatized[:15]
})
print('Stemming vs Lemmatization comparison:')
comparison

In [ ]:
# Step 6: Word Frequency Analysis
fdist = FreqDist(lemmatized)
print('Top 15 most common words:')
for word, freq in fdist.most_common(15):
    print(f'  {word:15s} : {freq}')

# Plot word frequencies
fig, ax = plt.subplots(figsize=(10, 6))
top_words = fdist.most_common(15)
words_plot = [w for w, _ in top_words]
freqs_plot = [f for _, f in top_words]
ax.barh(words_plot[::-1], freqs_plot[::-1], color='steelblue', edgecolor='black')
ax.set_xlabel('Frequency', fontsize=12)
ax.set_title('Top 15 Word Frequencies', fontsize=14, fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Step 7: Generate WordCloud
wordcloud = WordCloud(
    width=800, height=400,
    background_color='white',
    colormap='viridis',
    max_words=100,
    stopwords=stop_words,
    collocations=False
).generate(' '.join(lemmatized))

fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(wordcloud, interpolation='bilinear')
ax.axis('off')
ax.set_title('Word Cloud of Text Corpus', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Step 8: N-gram Analysis — Bigrams and Trigrams
bigram_list = list(bigrams(lemmatized))
trigram_list = list(trigrams(lemmatized))

bigram_freq = FreqDist(bigram_list)
trigram_freq = FreqDist(trigram_list)

print('Top 10 Bigrams:')
for bg, freq in bigram_freq.most_common(10):
    print(f'  {" ".join(bg):30s} : {freq}')

print('\nTop 10 Trigrams:')
for tg, freq in trigram_freq.most_common(10):
    print(f'  {" ".join(tg):40s} : {freq}')

In [ ]:
# Step 9: Sentiment Analysis — TextBlob
test_sentences = [
    'Data science is an amazing and exciting field with tremendous opportunities!',
    'The project was terrible and the results were completely disappointing.',
    'Python is a programming language used for data analysis.',
    'I absolutely love working with machine learning algorithms and seeing great results!',
    'The documentation is poor and the code is extremely confusing and frustrating.'
]

print('TEXTBLOB SENTIMENT ANALYSIS')
print('=' * 70)
print(f'{"Text":<55} {"Polarity":>8} {"Subjectivity":>12}')
print('-' * 75)

for text in test_sentences:
    blob = TextBlob(text)
    short = text[:50] + '...' if len(text) > 50 else text
    print(f'{short:<55} {blob.sentiment.polarity:>+8.3f} {blob.sentiment.subjectivity:>12.3f}')

In [ ]:
# Step 10: Sentiment Analysis — VADER
analyzer = SentimentIntensityAnalyzer()

print('VADER SENTIMENT ANALYSIS')
print('=' * 85)
print(f'{"Text":<45} {"Neg":>5} {"Neu":>5} {"Pos":>5} {"Compound":>8}')
print('-' * 75)

for text in test_sentences:
    scores = analyzer.polarity_scores(text)
    short = text[:42] + '...' if len(text) > 42 else text
    print(f'{short:<45} {scores["neg"]:>5.3f} {scores["neu"]:>5.3f} '
          f'{scores["pos"]:>5.3f} {scores["compound"]:>+8.3f}')

In [ ]:
# Step 11: Summary
print('=' * 60)
print('TEXT ANALYTICS PIPELINE — COMPLETED')
print('=' * 60)
print()
print('Steps performed:')
print('  1. Sentence and word tokenization')
print('  2. Lowercasing')
print('  3. Punctuation removal')
print('  4. Stop word removal')
print('  5. Stemming (Porter) and Lemmatization (WordNet)')
print('  6. Word frequency analysis with visualization')
print('  7. WordCloud generation')
print('  8. Bigram and Trigram analysis')
print('  9. Sentiment analysis with TextBlob (polarity + subjectivity)')
print('  10. Sentiment analysis with VADER (compound score)')

## Dataset Description

- **Corpus:** Custom text about data science (for demonstration)
- **Sentiment test:** 5 sentences ranging from positive to negative

## Conclusion

Complete text analytics pipeline successfully demonstrated. The preprocessing steps (tokenization, lowercasing, punctuation/stop word removal, stemming/lemmatization) transform raw text into clean, analyzable tokens. Word frequency and WordCloud provide visual understanding of key themes. N-gram analysis reveals common word pairs and phrases. Both TextBlob and VADER effectively capture sentiment — TextBlob provides polarity and subjectivity, while VADER offers a more nuanced breakdown with negative, neutral, positive, and compound scores.